<a href="https://colab.research.google.com/github/Perry-wang-ab741/114-2-ProgramingLanguage/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai

In [2]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [19]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [21]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make predictions or decisions.


In [22]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1V65JtvOG8CsP5rSpCoZ5WjTfMhw1i29N4pv7wW-D3sY/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [23]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [24]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:

        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [25]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績： 95
已記錄：日期: 2026-04-04, 科目: 國文, 成績: 95

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：80
已記錄：日期: 2026-04-04, 科目: 英文, 成績: 80

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：60
已記錄：日期: 2026-04-04, 科目: 數學, 成績: 60

請輸入科目（或輸入 'q' 停止）：q


In [26]:
new_grades

[['2026-04-04', '國文', 95], ['2026-04-04', '英文', 80], ['2026-04-04', '數學', 60]]

In [29]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，根據您提供的學生三科成績，以下為您整理的簡單摘要與常見迷思：\n\n---\n\n### **學生單次成績摘要 (2026-04-04)**\n\n**評量日期：** 2026年04月04日\n\n**科目與成績：**\n\n*   **國文：** 95分\n*   **英文：** 80分\n*   **數學：** 60分\n\n**總結：**\n\n這份成績記錄顯示學生在2026年04月04日進行了國文、英文、數學三門科目的評量。分數分佈介於60分至95分之間。其中，國文科目取得95分，是本次評量中得分最高的科目；英文科目獲得80分；數學科目則為60分。這是一次單一時間點的成績記錄。\n\n---\n\n### **關於學生學業成績的常見迷思整理 (不評分，只做總結)**\n\n學業成績是評量學習成果的一個重要工具，但圍繞著它也有許多常見的誤解。以下列出一些與成績相關的迷思：\n\n1.  **迷思一：單一成績能完全定義學生能力。**\n    *   **澄清：** 成績是特定時間點、針對特定學習內容的評量結果，它反映了學生在該次評量中的表現。然而，它不能全面反映一個人的潛力、努力程度、多元智慧（如創造力、人際智能）或個人特質。\n2.  **迷思二：成績是衡量「聰明才智」的唯一標準。**\n    *   **澄清：** 學術成績確實與某些認知能力和學習成果相關，但「聰明才智」是多面向的。解決問題的能力、批判性思維、實踐技能、情緒智商、藝術天賦等，這些不一定能完全透過傳統的學業分數來體現。\n3.  **迷思三：低分代表學生不努力或沒天賦。**\n    *   **澄清：** 分數低的原因可能是多種因素的結果，例如：學習方法不當、對該科目缺乏興趣、外部壓力、身體狀況不佳、考試當天表現失常，甚至與教學方式的契合度等，不應簡單歸因於努力或天賦。\n4.  **迷思四：高分就等於對知識有深入理解。**\n    *   **澄清：** 有時學生可能透過記憶、大量的練習或應試技巧取得高分，但這不一定代表他們對知識有融會貫通的理解，或是能將所學應用到實際問題中。真正的學習包含理解、分析、綜合和應用能力。\n5.  **迷思五：成績是絕對固定不變的。**\n    *   **澄清：** 學習是一個持續的過程，學生的成績會隨著學習投入、方法調整、成熟度和經驗的累

In [31]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：90
已記錄：日期: 2026-04-04, 科目: 國文, 成績: 90

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：85
已記錄：日期: 2026-04-04, 科目: 英文, 成績: 85

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：65
已記錄：日期: 2026-04-04, 科目: 數學, 成績: 65

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的學生單科成績所做的簡單摘要與常見迷思整理，全程不對學生成績進行任何評價或建議。

---

### 學生成績摘要

這是一份單一學生在特定日期（2026-04-04）的三個科目（國文、英文、數學）的成績記錄。

*   **科目與分數：**
    *   國文：90分
    *   英文：85分
    *   數學：65分
*   **分數範圍：** 這三科成績的分數範圍從65分到90分。
*   **平均分數：** 三科的平均分數為80分。

這份資料提供了一個學生在該時間點，這三個學科表現的數字化概況。

---

### 常見成績迷思整理（不評分，只做總結）

成績在教育評量中扮演重要角色，但人們對於成績常有一些誤解。以下整理幾個常見的迷思：

1.  **迷思一：成績是衡量學生一切能力的唯一標準。**
    *   **事實：** 成績主要反映學生在特定學術領域的知識掌握與應試能力，而非其全部潛力、創造力、解決問題能力、情商或人格特質。許多重要能力是難以透過標準化成績來衡量的。

2.  **迷思二：單一科目或某次的成績低落，代表學生不夠努力或沒有天賦。**
    *   **事實：** 單次的成績表現可能受到多種因素影響，例如考試當天的身心狀態、對特定單